In [3]:
import os

import pandas as pd
import numpy as np

import functions.cnn_helpers as help_funcs

# DNA MERFISH data (Su et al. 2020)

## Read distance info and save as arrays

In [25]:
dist_data = pd.read_csv('../data/Su_2020/selected_structures_95_1000_nm_thr.tsv', sep='\t')
transc_data = pd.read_csv('../data/Su_2020/transcriptional_data_95_1000_nm_thr.tsv', sep='\t')

dataset = pd.merge(dist_data, transc_data.iloc[:, :2], how='left', on='Chromosome copy number')

n_loci = 38

### Iterate through all structures and save them as an image

In [28]:
for id, row in dataset.iterrows():
    
    struct_id = int(row['Chromosome copy number'])
    state = int(row['BACH1'])
    if state == 0:
        state_save = 'inactive'
    else:
        state_save = 'active'
    
    dists = row.iloc[1 : -1]
    
    locus_i = 0
    dist_i = np.zeros((n_loci, n_loci))
    
    for i in range(0, n_loci):
        for j in range(i + 1, n_loci):
            dist_i[i,j] = dists.iloc[locus_i]
            
            locus_i += 1
    
    dist_i = dist_i + dist_i.T
    
    
    dist_i = pd.DataFrame(dist_i)
    dist_i.to_csv(f"../data/Su_2020/distance_maps_95_1000/{state_save}/structure_{struct_id}.tsv", sep='\t', header = False, index=False)

## Create splits and save them before (.png) and after (.tsv) normalization

In [29]:
help_funcs.kfold_splits(src_folder = '../data/Su_2020/distance_maps_95_1000/', dst_folder = '../data/Su_2020/CNN_splits_balanced_95_1000')

Creating fold 1/5...
Train size: 1556, Validation size: 638, Test size: 638
Creating fold 2/5...
Train size: 1556, Validation size: 638, Test size: 638
Creating fold 3/5...
Train size: 1556, Validation size: 638, Test size: 638
Creating fold 4/5...
Train size: 1556, Validation size: 638, Test size: 638
Creating fold 5/5...
Train size: 1556, Validation size: 638, Test size: 638
Created 5 stratified splits under '../data/Su_2020/CNN_splits_balanced_95_1000'.


In [5]:
samples = []

# Subfolders are class labels
classes = sorted(
    [
        class_i
        for class_i in os.listdir('../data/Su_2020/CNN_splits_balanced_zscore/fold_0/test')
        if os.path.isdir(os.path.join('../data/Su_2020/CNN_splits_balanced_zscore/fold_0/test', class_i))
    ],
    reverse=True,
)
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}

for label in os.listdir('../data/Su_2020/CNN_splits_balanced_zscore/fold_0/test'):
    class_dir = os.path.join('../data/Su_2020/CNN_splits_balanced_zscore/fold_0/test', label)
    if not os.path.isdir(class_dir):
        continue

    for fname in os.listdir(class_dir):
        if fname.endswith(".tsv"):
            path = os.path.join(class_dir, fname)
            samples.append((path, int(class_to_idx[label])))


# DNA MERFISH data (Su et al. 2020) - simulated dataset

## Read matrices, fill missing triangles and convert from Amber A.U. to real physical nanometers

In [55]:
transc_data = pd.read_csv('../data/Su_2020/simulated_dataset/structures_transcription_data.tsv', sep='\t')
#transc_data = transc_data[['Transcription', 'structure']]

In [56]:
transc_data

,1-2,1-3,1-4,1-5,1-6,1-7,1-8,1-9,1-10,2-3,...,7-8,7-9,7-10,8-9,8-10,9-10,Chromosome copy number,Transcription,dist_index,structure
0,7.0,7.0,7.0,22.0,37.0,7.0,22.0,7.0,7.0,7.0,...,22.0,22.0,7.0,22.0,7.0,22.0,1091,0,1,0
1,7.0,22.0,127.0,127.0,112.0,232.0,157.0,187.0,217.0,22.0,...,97.0,52.0,67.0,37.0,112.0,82.0,7454,1,2,813
2,7.0,37.0,97.0,52.0,52.0,142.0,157.0,142.0,217.0,37.0,...,22.0,157.0,82.0,172.0,67.0,202.0,7780,0,3,1412
3,7.0,37.0,157.0,97.0,127.0,112.0,67.0,127.0,112.0,37.0,...,97.0,82.0,202.0,67.0,187.0,232.0,1654,0,4,1772
4,7.0,52.0,82.0,67.0,157.0,187.0,157.0,277.0,202.0,37.0,...,67.0,157.0,67.0,157.0,82.0,82.0,10368,1,5,2493
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3010,322.0,337.0,352.0,202.0,217.0,112.0,37.0,52.0,7.0,52.0,...,127.0,82.0,97.0,97.0,52.0,52.0,11786,0,2999,1499436
3011,337.0,127.0,37.0,142.0,67.0,142.0,232.0,187.0,247.0,367.0,...,142.0,97.0,112.0,247.0,187.0,112.0,9630,1,3000,1499987
3012,337.0,142.0,172.0,187.0,202.0,202.0,187.0,322.0,202.0,202.0,...,157.0,187.0,202.0,172.0,67.0,142.0,3224,0,3001,1500090
3013,367.0,262.0,247.0,112.0,247.0,217.0,112.0,127.0,172.0,142.0,...,112.0,82.0,187.0,52.0,82.0,127.0,8853,1,3002,1500772


In [57]:
transc_data.head()

,1-2,1-3,1-4,1-5,1-6,1-7,1-8,1-9,1-10,2-3,...,7-8,7-9,7-10,8-9,8-10,9-10,Chromosome copy number,Transcription,dist_index,structure
0,7.0,7.0,7.0,22.0,37.0,7.0,22.0,7.0,7.0,7.0,...,22.0,22.0,7.0,22.0,7.0,22.0,1091,0,1,0
1,7.0,22.0,127.0,127.0,112.0,232.0,157.0,187.0,217.0,22.0,...,97.0,52.0,67.0,37.0,112.0,82.0,7454,1,2,813
2,7.0,37.0,97.0,52.0,52.0,142.0,157.0,142.0,217.0,37.0,...,22.0,157.0,82.0,172.0,67.0,202.0,7780,0,3,1412
3,7.0,37.0,157.0,97.0,127.0,112.0,67.0,127.0,112.0,37.0,...,97.0,82.0,202.0,67.0,187.0,232.0,1654,0,4,1772
4,7.0,52.0,82.0,67.0,157.0,187.0,157.0,277.0,202.0,37.0,...,67.0,157.0,67.0,157.0,82.0,82.0,10368,1,5,2493


In [59]:
original_mats_folder = '../data/Su_2020/simulated_dataset/matrices_AU_upper_diagonal'

dist_mats = os.listdir(original_mats_folder)

In [62]:
# Go through each matrix, process and save according to transcriptional state
scaling_factor = 0.0237699544

for mat_i_file in dist_mats:
    mat_i = pd.read_csv(f'{original_mats_folder}/{mat_i_file}', sep='\t', header=None).to_numpy() / scaling_factor / 10
    
    mat_i[np.isnan(mat_i)] = 0
    mat_i = mat_i + np.transpose(mat_i)

    frame = int(mat_i_file.split('_')[-1].split('.')[0])
    frame_transc = transc_data[transc_data['structure'] == frame]['Transcription'].values[0]
     
    if frame_transc == 0:
        state_save = 'inactive'
    else:
        state_save = 'active'
    
    mat_i = pd.DataFrame(mat_i)
    mat_i.to_csv(f"../data/Su_2020/simulated_dataset/matrices_transformed_classified/{state_save}/structure_{frame}.tsv", sep='\t', header = False, index=False)

## Create splits and save them before (.png) and after (.tsv) normalization

In [63]:
help_funcs.kfold_splits(src_folder = '../data/Su_2020/simulated_dataset/matrices_transformed_classified/', dst_folder = '../data/Su_2020/simulated_dataset/CNN_splits_balanced_95_1000')

Creating fold 1/5...
Train size: 1466, Validation size: 603, Test size: 603
Creating fold 2/5...
Train size: 1466, Validation size: 603, Test size: 603
Creating fold 3/5...
Train size: 1466, Validation size: 603, Test size: 603
Creating fold 4/5...
Train size: 1466, Validation size: 603, Test size: 603
Creating fold 5/5...
Train size: 1466, Validation size: 603, Test size: 603
Created 5 stratified splits under '../data/Su_2020/simulated_dataset/CNN_splits_balanced_95_1000'.


In [66]:
len(os.listdir('../data/Su_2020/simulated_dataset/CNN_splits_balanced_95_1000_zscore/fold_4/test/inactive'))

359

In [119]:
samples = []

# Subfolders are class labels
classes = sorted(
    [
        class_i
        for class_i in os.listdir('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test')
        if os.path.isdir(os.path.join('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test', class_i))
    ],
    reverse=True,
)
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}

for label in os.listdir('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test'):
    class_dir = os.path.join('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test', label)
    if not os.path.isdir(class_dir):
        continue

    for fname in os.listdir(class_dir):
        if fname.endswith(".tsv"):
            path = os.path.join(class_dir, fname)
            samples.append((path, int(class_to_idx[label])))


In [120]:
samples

[('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test/inactive/structure_1248573.tsv',
  0),
 ('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test/inactive/structure_666262.tsv',
  0),
 ('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test/inactive/structure_846540.tsv',
  0),
 ('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test/inactive/structure_23567.tsv',
  0),
 ('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test/inactive/structure_314629.tsv',
  0),
 ('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test/inactive/structure_995297.tsv',
  0),
 ('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test/inactive/structure_772001.tsv',
  0),
 ('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test/inactive/structure_625011.tsv',
  0),
 ('../data/Su_2020/simulated_dataset/CNN_splits_balanced_zscore/fold_0/test/inactive/str

# ORCA data (Mateo et al. 2020)

## Read distance info and save as arrays

In [4]:
dist_data = pd.read_csv('../data/Mateo_2019/selected_structures_25_5000_filtered.tsv', sep='\t')
transc_data = pd.read_csv('../data/Mateo_2019/transcriptional_data_25_5000_filtered.tsv', sep='\t')

dataset = pd.merge(dist_data, transc_data.iloc[:, :2], how='left', on='cellNumber')

n_loci = 52

In [5]:
dataset

,cellNumber,1-2,1-3,1-4,1-5,1-6,1-7,1-8,1-9,1-10,...,48-50,48-51,48-52,49-50,49-51,49-52,50-51,50-52,51-52,Abd-A_Intron
0,1.0,350.552643,701.105652,742.068665,666.562439,836.412598,821.212219,917.282715,834.493896,899.518677,...,57.293232,85.939850,114.586464,28.646616,57.293232,85.939850,28.646616,57.293232,28.646616,0
1,2.0,244.073929,156.705215,227.523438,298.594910,284.360779,319.927277,374.712128,263.243683,307.865479,...,158.210205,205.904953,89.207565,190.698318,243.541779,129.752823,129.892990,114.832573,134.459518,0
2,3.0,262.981628,149.599136,223.714142,298.312103,284.214844,319.122009,373.181580,263.169769,307.152435,...,148.777985,164.663910,56.323582,173.666397,188.108734,68.595428,74.033485,109.178902,135.220947,0
3,4.0,57.026531,114.053062,158.372696,99.316757,381.578186,285.982574,245.477722,143.316086,131.084579,...,95.183998,142.776352,134.868652,47.591965,95.184311,87.573639,47.592350,40.972393,14.776049,0
4,5.0,96.074387,248.331451,431.943512,365.492401,312.008545,323.191193,344.080688,373.049774,408.382874,...,27.680204,41.520115,55.360455,13.839910,27.679819,41.520161,13.839910,27.680250,13.840343,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52967,54360.0,102.750069,205.500137,308.250214,411.000397,210.793076,504.672241,322.819763,419.980103,519.280396,...,115.259415,608.861206,1105.478760,114.214897,597.610779,1093.241211,496.962738,993.925537,496.962769,1
52968,54361.0,195.376205,331.793182,311.332672,312.916718,343.966980,439.232422,1014.994324,715.538147,457.085999,...,166.204132,209.301727,253.618362,83.102074,138.592422,221.422577,91.762802,218.049500,295.440369,0
52969,54362.0,84.795517,169.591034,254.386398,387.011017,332.686523,498.608093,822.776733,443.369690,369.567047,...,283.546692,574.693970,484.069580,62.116104,796.917725,689.178223,857.717712,749.528992,234.870514,0
52970,54363.0,45.831791,91.663933,227.901001,305.233826,403.958008,567.012085,793.995728,1044.012451,871.298706,...,57.775463,139.123123,133.248871,82.571701,165.143433,149.427689,82.571732,82.203911,69.444191,0


In [6]:
dataset['Abd-A_Intron'].value_counts()

Abd-A_Intron
0    41833
1    11139
Name: count, dtype: int64

### Iterate through all structures and save them as an image

In [9]:
for id, row in dataset.iterrows():
    
    struct_id = int(row['cellNumber'])
    state = int(row['Abd-A_Intron'])
    if state == 0:
        state_save = 'inactive'
    else:
        state_save = 'active'
    
    dists = row.iloc[1 : -1]
    
    locus_i = 0
    dist_i = np.zeros((n_loci, n_loci))
    
    for i in range(0, n_loci):
        for j in range(i + 1, n_loci):
            dist_i[i,j] = dists.iloc[locus_i]
            
            locus_i += 1
    
    dist_i = dist_i + dist_i.T
    
    
    dist_i = pd.DataFrame(dist_i)
    dist_i.to_csv(f"../data/Mateo_2019/distance_maps_25_5000_filtered/{state_save}/structure_{struct_id}.tsv", sep='\t', header = False, index=False)

## Create splits and save them before (.png) and after (.tsv) normalization

In [10]:
help_funcs.kfold_splits(src_folder = '../data/Mateo_2019/distance_maps_25_5000_filtered/', dst_folder = '../data/Mateo_2019/CNN_splits_balanced_25_5000_filtered', test_size=0.2, val_size=0.25)

Creating fold 1/5...
Train size: 13366, Validation size: 10595, Test size: 10595
Creating fold 2/5...
Train size: 13366, Validation size: 10595, Test size: 10595
Creating fold 3/5...
Train size: 13366, Validation size: 10595, Test size: 10595
Creating fold 4/5...
Train size: 13366, Validation size: 10595, Test size: 10595
Creating fold 5/5...
Train size: 13366, Validation size: 10595, Test size: 10595
Created 5 stratified splits under '../data/Mateo_2019/CNN_splits_balanced_25_5000_filtered'.


In [ ]:
# Also try with unbalanced dataset
#help_funcs.kfold_splits_unbalanced(src_folder = '../data/Mateo_2019/distance_maps_75_5000_filtered/', dst_folder = '../data/Mateo_2019/CNN_splits_unbalanced_75_5000_filtered', test_size=0.2, val_size=0.25)

Creating fold 1/5...
Train size: 1376, Validation size: 459, Test size: 459
Creating fold 2/5...
Train size: 1376, Validation size: 459, Test size: 459
Creating fold 3/5...
Train size: 1376, Validation size: 459, Test size: 459
Creating fold 4/5...
Train size: 1376, Validation size: 459, Test size: 459
Creating fold 5/5...
Train size: 1376, Validation size: 459, Test size: 459
Created 5 stratified splits under '../data/Mateo_2019/CNN_splits_unbalanced_75_5000_filtered'.


In [22]:
samples = []

# Subfolders are class labels
classes = sorted(
    [
        class_i
        for class_i in os.listdir('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test')
        if os.path.isdir(os.path.join('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test', class_i))
    ],
    reverse=True,
)
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}

for label in os.listdir('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test'):
    class_dir = os.path.join('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test', label)
    if not os.path.isdir(class_dir):
        continue

    for fname in os.listdir(class_dir):
        if fname.endswith(".tsv"):
            path = os.path.join(class_dir, fname)
            samples.append((path, int(class_to_idx[label])))


In [23]:
samples

[('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test/inactive/structure_24585.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test/inactive/structure_12751.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test/inactive/structure_45609.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test/inactive/structure_10146.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test/inactive/structure_27926.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test/inactive/structure_33953.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test/inactive/structure_49741.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test/inactive/structure_17615.tsv',
  0),
 ('../data/Mateo_2019/CNN_splits_balanced_50_5000_filtered_zscore/fold_0/test/in